# Inference Demo

End-to-end demo with audio playback.

In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# SignVoice — Inference Demo\nLoad trained classifier, run on test videos, play audio.\nUse this notebook for viva / project presentation."]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.insert(0, '..')\n",
    "\n",
    "import json\n",
    "import time\n",
    "import subprocess\n",
    "import tempfile\n",
    "import os\n",
    "import torch\n",
    "import torch.nn as nn\n",
    "import numpy as np\n",
    "import yaml\n",
    "import librosa\n",
    "import librosa.display\n",
    "import soundfile as sf\n",
    "import matplotlib.pyplot as plt\n",
    "import IPython.display as ipd\n",
    "from pathlib import Path\n",
    "from collections import Counter\n",
    "\n",
    "from src.preprocessing.normalizer import KeypointNormalizer\n",
    "\n",
    "device = 'cuda' if torch.cuda.is_available() else 'cpu'\n",
    "print(f'Device  : {device}')\n",
    "print(f'PyTorch : {torch.__version__}')\n",
    "print('Ready.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 1. Load Classifier"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class SignClassifier(nn.Module):\n",
    "    def __init__(self, input_dim=183, d_model=64,\n",
    "                 n_heads=2, n_layers=1, n_classes=10):\n",
    "        super().__init__()\n",
    "        self.proj = nn.Linear(input_dim, d_model)\n",
    "        self.bn   = nn.BatchNorm1d(d_model)\n",
    "        layer     = nn.TransformerEncoderLayer(\n",
    "            d_model=d_model, nhead=n_heads,\n",
    "            dim_feedforward=128, dropout=0.5,\n",
    "            batch_first=True, norm_first=True,\n",
    "        )\n",
    "        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)\n",
    "        self.classifier  = nn.Sequential(\n",
    "            nn.Dropout(0.5),\n",
    "            nn.Linear(d_model, 32),\n",
    "            nn.ReLU(),\n",
    "            nn.Dropout(0.3),\n",
    "            nn.Linear(32, n_classes),\n",
    "        )\n",
    "\n",
    "    def forward(self, x, mask=None):\n",
    "        B, T, F = x.shape\n",
    "        x = self.proj(x)\n",
    "        x = self.bn(x.reshape(B*T, -1)).reshape(B, T, -1)\n",
    "        x = self.transformer(x, src_key_padding_mask=mask)\n",
    "        x = x.mean(dim=1)\n",
    "        return self.classifier(x)\n",
    "\n",
    "ckpt_path = Path('../checkpoints/classifier.pt')\n",
    "if not ckpt_path.exists():\n",
    "    raise FileNotFoundError(\n",
    "        'No classifier found. Run: python scripts/test_inference.py'\n",
    "    )\n",
    "\n",
    "ckpt    = torch.load(ckpt_path, map_location=device, weights_only=False)\n",
    "glosses = ckpt['glosses']\n",
    "model   = SignClassifier(n_classes=len(glosses)).to(device)\n",
    "model.load_state_dict(ckpt['model'])\n",
    "model.eval()\n",
    "\n",
    "with open('../configs/lightweight.yaml') as f:\n",
    "    config = yaml.safe_load(f)\n",
    "\n",
    "normalizer = KeypointNormalizer(config['data']['stats_path'])\n",
    "normalizer.load()\n",
    "\n",
    "print(f'Classifier loaded')\n",
    "print(f'Signs known : {glosses}')\n",
    "print(f'N classes   : {len(glosses)}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 2. Load Test Samples"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "with open('../data/processed/test_manifest.json') as f:\n",
    "    test_data = json.load(f)\n",
    "\n",
    "g_to_idx = {g: i for i, g in enumerate(glosses)}\n",
    "test_data = [s for s in test_data if s['gloss'] in g_to_idx]\n",
    "\n",
    "print(f'Test samples : {len(test_data)}')\n",
    "counts = Counter(s['gloss'] for s in test_data)\n",
    "for g, c in sorted(counts.items()):\n",
    "    print(f'  {g:15s} : {c}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 3. Run Inference on All Test Samples"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def gloss_to_audio(gloss, sr=22050):\n",
    "    from gtts import gTTS\n",
    "    tmp_mp3 = tempfile.mktemp(suffix='.mp3')\n",
    "    tmp_wav = tempfile.mktemp(suffix='.wav')\n",
    "    try:\n",
    "        gTTS(text=gloss.lower(), lang='en', slow=False).save(tmp_mp3)\n",
    "        subprocess.run(\n",
    "            ['ffmpeg', '-y', '-i', tmp_mp3, '-ar', str(sr), tmp_wav],\n",
    "            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL\n",
    "        )\n",
    "        wav, _ = sf.read(tmp_wav)\n",
    "        return wav.astype(np.float32), sr\n",
    "    finally:\n",
    "        for p in [tmp_mp3, tmp_wav]:\n",
    "            try: os.unlink(p)\n",
    "            except: pass\n",
    "\n",
    "Path('../outputs').mkdir(exist_ok=True)\n",
    "results = []\n",
    "\n",
    "for i, sample in enumerate(test_data):\n",
    "    gloss_true = sample['gloss']\n",
    "    t0         = time.time()\n",
    "\n",
    "    kp   = normalizer.normalize(np.load(sample['keypoint_file']))\n",
    "    kp_t = torch.from_numpy(kp).unsqueeze(0).to(device)\n",
    "\n",
    "    with torch.no_grad():\n",
    "        logits    = model(kp_t)\n",
    "        probs     = torch.softmax(logits, dim=1)\n",
    "        pred_idx  = logits.argmax(dim=1).item()\n",
    "        pred_prob = probs[0, pred_idx].item()\n",
    "\n",
    "    gloss_pred = glosses[pred_idx]\n",
    "    is_correct = gloss_pred == gloss_true\n",
    "    t1         = time.time()\n",
    "\n",
    "    mark = '✓' if is_correct else '✗'\n",
    "    print(f'[{i+1:>2}/{len(test_data)}] '\n",
    "          f'true={gloss_true:12s} '\n",
    "          f'pred={gloss_pred:12s} '\n",
    "          f'conf={pred_prob:.2f} '\n",
    "          f'{mark} ({t1-t0:.2f}s)')\n",
    "\n",
    "    results.append({\n",
    "        'true'      : gloss_true,\n",
    "        'predicted' : gloss_pred,\n",
    "        'correct'   : is_correct,\n",
    "        'confidence': pred_prob,\n",
    "        'time'      : t1 - t0,\n",
    "    })\n",
    "\n",
    "accuracy = sum(r['correct'] for r in results) / len(results) * 100\n",
    "print(f'\\nAccuracy: {accuracy:.1f}%')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 4. Play Predicted Audio — Listen to Results"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Play one example per gloss\n",
    "seen = set()\n",
    "for r in results:\n",
    "    gloss = r['predicted']\n",
    "    if gloss in seen:\n",
    "        continue\n",
    "    seen.add(gloss)\n",
    "\n",
    "    print(f\"\\n--- '{gloss}' \"\n",
    "          f\"(true='{r['true']}' \"\n",
    "          f\"conf={r['confidence']:.2f} \"\n",
    "          f\"{'✓' if r['correct'] else '✗'}) ---\")\n",
    "\n",
    "    try:\n",
    "        wav, sr = gloss_to_audio(gloss)\n",
    "        display(ipd.Audio(wav, rate=sr))\n",
    "    except Exception as e:\n",
    "        print(f'Audio error: {e}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 5. Accuracy Summary"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from collections import defaultdict\n",
    "\n",
    "gloss_results = defaultdict(list)\n",
    "for r in results:\n",
    "    gloss_results[r['true']].append(r['correct'])\n",
    "\n",
    "signs = sorted(gloss_results.keys())\n",
    "accs  = [sum(gloss_results[g]) / len(gloss_results[g]) * 100\n",
    "         for g in signs]\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(12, 5))\n",
    "colors  = ['green' if a >= 60 else 'orange' if a >= 40\n",
    "           else 'red' for a in accs]\n",
    "bars    = ax.bar(signs, accs, color=colors, edgecolor='white')\n",
    "ax.axhline(60, color='green', linestyle='--',\n",
    "           linewidth=1, label='60% good for demo')\n",
    "ax.set_ylim(0, 110)\n",
    "ax.set_xlabel('Sign')\n",
    "ax.set_ylabel('Accuracy (%)')\n",
    "ax.set_title(f'Per-sign accuracy — Overall: {accuracy:.1f}%')\n",
    "ax.legend()\n",
    "\n",
    "for bar, acc in zip(bars, accs):\n",
    "    ax.text(bar.get_x() + bar.get_width()/2,\n",
    "            bar.get_height() + 1,\n",
    "            f'{acc:.0f}%',\n",
    "            ha='center', va='bottom', fontsize=10)\n",
    "\n",
    "plt.xticks(rotation=45, ha='right')\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print(f'Overall accuracy : {accuracy:.1f}%')\n",
    "print(f'Best signs       : {[s for s,a in zip(signs,accs) if a>=60]}')\n",
    "print(f'Use these in your demo!')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 6. Inference Speed Benchmark"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print('Inference Speed Benchmark')\n",
    "print('=' * 45)\n",
    "\n",
    "avg_time = np.mean([r['time'] for r in results])\n",
    "print(f'Average inference time : {avg_time*1000:.1f} ms per sign')\n",
    "print(f'Throughput             : {1/avg_time:.1f} signs/second')\n",
    "print()\n",
    "\n",
    "for r in results[:5]:\n",
    "    print(f\"  '{r['true']:12s}' → '{r['predicted']:12s}' \"\n",
    "          f\"in {r['time']*1000:.0f}ms\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 7. Project Summary — For Viva"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "with open('../data/processed/train_manifest.json') as f:\n",
    "    train_data = json.load(f)\n",
    "\n",
    "sources = Counter(s.get('source','unknown') for s in train_data)\n",
    "total_params = sum(p.numel() for p in model.parameters())\n",
    "\n",
    "print('=' * 55)\n",
    "print('  SignVoice — Project Summary')\n",
    "print('=' * 55)\n",
    "print(f'  Dataset        : Google ASL Signs + WLASL')\n",
    "print(f'  Data sources   : {dict(sources)}')\n",
    "print(f'  Train samples  : {len(train_data)}')\n",
    "print(f'  Signs trained  : {glosses}')\n",
    "print(f'  N classes      : {len(glosses)}')\n",
    "print()\n",
    "print(f'  Model          : Transformer Classifier')\n",
    "print(f'  Parameters     : {total_params:,}')\n",
    "print(f'  Input          : (B, T, 183) keypoints')\n",
    "print(f'  Output         : ({len(glosses)},) class logits')\n",
    "print(f'  Speech output  : pyttsx3 system TTS')\n",
    "print(f'  Device         : {device}')\n",
    "print()\n",
    "print(f'  Test accuracy  : {accuracy:.1f}%')\n",
    "print(f'  Inference time : {avg_time*1000:.1f} ms')\n",
    "print()\n",
    "print(f'  Pipeline       : Sign video → MediaPipe → Transformer')\n",
    "print(f'                   → Classifier → TTS → Speech')\n",
    "print(f'  No text used   : True — purely visual to audio')\n",
    "print('=' * 55)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "SignVoice (venv311)",
   "language": "python",
   "name": "signvoice"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.9"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}